# Lab 3 - A Research Agent that Files to Google Docs

**Section 5 · Designing Effective Agents**  ·  **Estimated time:** 25–30 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

> **What you'll build:** an agent that web-searches a topic, writes a concise cited brief, then files that brief into **Google Docs** through a remote MCP server. This is your first agent that combines the built-in toolset (`web_search`, `web_fetch`, `write`) with an external **MCP connector** that takes real action in another product.

## Goal
Build a `Research Brief` agent that researches a topic, writes a well-cited brief, then creates a brand-new Google Doc containing it with citations carried through. You will learn to combine the built-in toolset with an external MCP connector in one agent, declare an `mcp_servers` entry and expose it via an `mcp_toolset`, drive a real side effect (a created Google Doc) from a session, and configure environment networking so both web tools and MCP servers work.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- A Google Docs connection configured in Claude Managed Agents Vaults. Paste `GOOGLE_DOCS_VAULT_ID` in the setup cell. The notebook can usually read the MCP URL from the vault credential; only paste `GOOGLE_DOCS_MCP_URL` if your vault has multiple MCP credentials.

> **Note on credentials:** This lab uses Claude Managed Agents Vaults for Google OAuth. The token stays in the vault; the session only receives the vault id.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ").strip()

print("Configure your Google Docs Managed Agents vault for this notebook.")
existing_vault_id = os.environ.get("GOOGLE_DOCS_VAULT_ID", "").strip()
if existing_vault_id.startswith("sk-ant-") or (existing_vault_id and not existing_vault_id.startswith("vlt_")):
    print("Clearing invalid GOOGLE_DOCS_VAULT_ID. It must be a vault id starting with 'vlt_', not an API key.")
    os.environ.pop("GOOGLE_DOCS_VAULT_ID", None)

if not os.environ.get("GOOGLE_DOCS_VAULT_ID"):
    os.environ["GOOGLE_DOCS_VAULT_ID"] = input(
        "Google Docs vault ID from Claude Console (vlt_...): "
    ).strip()

# Optional advanced override only. In the normal course path, leave this unset;
# the next Google Docs cell reads the MCP URL from the vault credential.
# os.environ["GOOGLE_DOCS_MCP_URL"] = "https://your-google-docs-mcp.example.com/mcp"

print("ANTHROPIC_API_KEY configured for this notebook session.")
print("GOOGLE_DOCS_VAULT_ID configured for this notebook session:", os.environ["GOOGLE_DOCS_VAULT_ID"])


In [ ]:
import os, sys
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Write a four-ingredient system prompt
The prompt follows the **role / constraints / tools / deliverable** structure. It tells the agent to research, write a cited brief, **then create a Google Doc** with it and report the URL.

In [ ]:
RESEARCH_BRIEF_GDOCS_SYSTEM = """\
ROLE
You are a research analyst. Your job is to research a topic the user
provides, write a concise, well-cited brief, then file that brief into
Google Docs.

CONSTRAINTS
- Always cite sources by URL inline, e.g. ([anthropic.com](https://anthropic.com)).
- Never invent numbers. If you can't verify a number, flag it as unverified.
- Keep the final brief concise.

TOOLS
You have web_search, web_fetch, write, and a "google_docs" MCP server.
Use web_search to discover sources, web_fetch to read them, write to draft
the brief, then call the google_docs MCP to create a new document.

DELIVERABLE
End every session by creating ONE Google Doc whose body is the finished
brief (exec summary, 3-5 cited paragraphs, a 'Sources' section). Report the
URL of the created Google Doc in your final message.
"""
print(RESEARCH_BRIEF_GDOCS_SYSTEM)

### Step 2 - Declare the Google Docs MCP server on the agent
Paste the vault id once in the setup cell. This step reads the Google Docs MCP URL from the vault credential when `GOOGLE_DOCS_MCP_URL` is not set, validates it before the API call, then creates the agent with the **built-in toolset** plus an `mcp_servers` entry for Google Docs exposed via an `mcp_toolset`. The session attaches the vault later with `vault_ids`.

In [ ]:
from urllib.parse import urlparse

GOOGLE_DOCS_VAULT_ID = os.environ.get("GOOGLE_DOCS_VAULT_ID", "").strip()
GOOGLE_DOCS_MCP_URL = os.environ.get("GOOGLE_DOCS_MCP_URL", "").strip()


def validate_mcp_url(url: str) -> None:
    """Catch missing or placeholder URLs before the API returns a generic 400."""
    parsed = urlparse(url)
    if not url or "REPLACE-ME" in url or parsed.scheme != "https" or not parsed.netloc:
        raise RuntimeError(
            "Set GOOGLE_DOCS_MCP_URL to a valid https MCP endpoint, or use a "
            "GOOGLE_DOCS_VAULT_ID whose credential contains an MCP server URL."
        )


def validate_vault_id(vault_id: str) -> None:
    """Catch common copy/paste mistakes before sessions.create."""
    if not vault_id or vault_id.startswith("vlt_REPLACE"):
        raise RuntimeError("Set GOOGLE_DOCS_VAULT_ID to your Claude Managed Agents vault id.")
    if vault_id.startswith("sk-ant-"):
        raise RuntimeError(
            "GOOGLE_DOCS_VAULT_ID currently contains an Anthropic API key. "
            "Paste the Google Docs vault id from Claude Console instead; it "
            "should start with 'vlt_'."
        )
    if not vault_id.startswith("vlt_"):
        raise RuntimeError(f"GOOGLE_DOCS_VAULT_ID should start with 'vlt_'. Got: {vault_id!r}")


def google_docs_mcp_url_from_vault(vault_id: str) -> str:
    """Read the MCP URL from the first Google Docs MCP OAuth credential."""
    credentials = list(client.beta.vaults.credentials.list(vault_id, betas=BETAS))
    mcp_credentials = [
        credential for credential in credentials
        if getattr(credential.auth, "type", None) == "mcp_oauth"
    ]
    google_credentials = [
        credential for credential in mcp_credentials
        if "google" in (
            f"{credential.display_name or ''} "
            f"{getattr(credential.auth, 'mcp_server_url', '')}"
        ).lower()
    ]

    if len(google_credentials) == 1:
        return google_credentials[0].auth.mcp_server_url
    if len(mcp_credentials) == 1:
        return mcp_credentials[0].auth.mcp_server_url

    names = [
        f"{credential.display_name or credential.id}: "
        f"{getattr(credential.auth, 'mcp_server_url', '<no mcp url>')}"
        for credential in mcp_credentials
    ]
    raise RuntimeError(
        "Could not uniquely identify the Google Docs MCP credential in "
        f"vault {vault_id}. Set GOOGLE_DOCS_MCP_URL explicitly. "
        f"Found MCP credentials: {names or 'none'}"
    )


validate_vault_id(GOOGLE_DOCS_VAULT_ID)

if not GOOGLE_DOCS_MCP_URL:
    GOOGLE_DOCS_MCP_URL = google_docs_mcp_url_from_vault(GOOGLE_DOCS_VAULT_ID)

validate_mcp_url(GOOGLE_DOCS_MCP_URL)
print("Google Docs MCP URL:", GOOGLE_DOCS_MCP_URL)

mcp_server = {"type": "url", "name": "google_docs", "url": GOOGLE_DOCS_MCP_URL}

agent = client.beta.agents.create(
    name="Research Brief (Google Docs)",
    model=MODEL,
    system=RESEARCH_BRIEF_GDOCS_SYSTEM,
    mcp_servers=[mcp_server],
    tools=[
        # Built-in toolset: web_search, web_fetch, write, read, bash, ...
        {"type": "agent_toolset_20260401"},
        # Expose the Google Docs MCP tools to the agent. Auth comes from the vault.
        {"type": "mcp_toolset",
         "mcp_server_name": "google_docs",
         "default_config": {"permission_policy": {"type": "always_allow"}}},
    ],
    betas=BETAS,
)
print("agent.id =", agent.id)


### Step 3 - Create a cloud environment that allows MCP + web
The agent needs to reach both the public web (for research) and the MCP server. Use **limited** networking with `allow_mcp_servers` on and valid hostname wildcard patterns for research. The API does not accept a bare `"*"` entry.

In [ ]:
RESEARCH_ALLOWED_HOSTS = [
    "*.com",
    "*.org",
    "*.net",
    "*.edu",
    "*.gov",
    "*.int",
    "*.io",
    "*.ai",
    "*.dev",
    "*.co",
    "*.de",
    "*.uk",
    "*.eu",
]

env = client.beta.environments.create(
    name="research-gdocs-env",
    config={"type": "cloud", "networking": {
        "type": "limited",
        "allow_mcp_servers": True,
        # Hostname wildcard patterns only; the API rejects a bare "*".
        "allowed_hosts": RESEARCH_ALLOWED_HOSTS,
    }},
    betas=BETAS,
)
print("env.id =", env.id)


### Step 4 - Run a session and watch the tools fire
Start a session pinned to this agent version, send the research request, and stream the run. Watch `web_search` / `web_fetch` / `write` fire as the agent works, then an `[mcp: ...]` call that creates the doc.

In [ ]:
TOPIC = "the state of small modular reactors in 2026"

GOOGLE_DOCS_VAULT_ID = os.environ.get("GOOGLE_DOCS_VAULT_ID", "").strip()
validate_vault_id(GOOGLE_DOCS_VAULT_ID)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=env.id,
    vault_ids=[GOOGLE_DOCS_VAULT_ID],
    title="SMRs 2026 -> Google Docs",
    betas=BETAS,
)
print("session.id =", session.id, "\n")

with client.beta.sessions.events.stream(session.id, betas=BETAS) as stream:
    client.beta.sessions.events.send(session.id, events=[{
        "type": "user.message",
        "content": [{"type": "text", "text": (
            f"Research {TOPIC}. Write a concise, cited brief, then "
            "create a Google Doc containing it and tell me the URL."
        )}],
    }], betas=BETAS)
    for event in stream:
        if event.type == "agent.message":
            for b in event.content:
                if b.type == "text":
                    print(b.text, end="", flush=True)
        elif event.type == "agent.tool_use":
            q = (event.input or {}).get("query", "")
            print(f"\n[tool: {event.name}({q[:60]})]")
        elif event.type == "agent.mcp_tool_use":
            print(f"\n[mcp: {event.name}]")
        elif event.type == "session.error":
            print(f"\n[ERROR] {event.error}")
        elif event.type == "session.status_idle":
            print("\n--- session idle ---")
            break

### Step 5 - Pull the result
The deliverable is the **Google Doc itself**, not a local file. The agent reports the new document's URL in its final `agent.message` above. Open that URL to confirm the brief landed with its citations intact.

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, [session.id], MODEL, betas=BETAS)


## Expected output
- An `agent.id`, `env.id`, and `session.id` line.
- Streamed `[tool: web_search]` and `[tool: web_fetch]` lines as it researches.
- A `[tool: write]` as it drafts the brief.
- An `[mcp: ...]` call that **creates the Google Doc**.
- A final message containing the **Google Doc URL**.
- Open the doc: a concise brief with an executive summary, cited analysis, and a Sources section listing the URLs actually consulted.

## Troubleshooting
- **400 on `agents.create`** → the MCP server URL is missing, a placeholder, or not a valid HTTPS MCP endpoint. Paste `GOOGLE_DOCS_VAULT_ID` in the setup cell and leave `GOOGLE_DOCS_MCP_URL` blank unless your vault has multiple MCP credentials.
- **400 on `sessions.create` with `invalid vault_id`** → `GOOGLE_DOCS_VAULT_ID` is not the vault id. If it starts with `sk-ant-`, you pasted the Anthropic API key into the vault field. Clear it with `os.environ.pop("GOOGLE_DOCS_VAULT_ID", None)`, rerun the setup cell, and paste the `vlt_...` id from Claude Console.
- **MCP auth fails** (401/403 from the connector) → your `GOOGLE_DOCS_VAULT_ID` is wrong, or the credential in the vault is expired/missing. Confirm the vault credential was created for the exact MCP server URL declared on the agent.
- **Web tools can't reach the internet** → the environment networking blocked them. Make sure `networking.type` is `limited` with hostname patterns like `"*.com"` in `allowed_hosts`; a bare `"*"` is rejected. Also set `allow_mcp_servers: True` so the connector is reachable.
- **Empty results / no brief** → the topic may be too narrow or the agent ran out of good sources. Broaden the topic, or add a follow-up turn: "Search more widely and try again." If the doc was never created, check the system prompt's deliverable section explicitly names the `google_docs` MCP.
- **`agent.version` errors** → if your SDK build doesn't expose `agent.version`, drop it and pass `agent=agent.id` to `sessions.create`.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste:

> "Build me a Managed Agents agent called 'Research Brief (Google Docs)'. Use `claude-haiku-4-5-20251001` and a four-section system prompt (role / constraints / tools / deliverable). Enable the full agent toolset and declare a Google Docs MCP server from the `GOOGLE_DOCS_MCP_URL` env var. Tell the agent to research a topic, write a concise cited brief, then create a Google Doc with it and report the URL."

Then:

> "Create a limited-networking cloud environment that allows MCP servers and the open web, start a session, and ask for a brief on the state of small modular reactors in 2026. Stream the run and print the Google Doc URL it returns."

Compare what Claude Code writes to the cells above.

## Stretch
- **Add a length budget.** Add a constraint such as "Keep the brief under 500 words" and a final `bash` word-count check before the doc is created.
- **List sources at the end.** Require a dedicated "Sources" section at the bottom of the Google Doc enumerating every URL the agent actually fetched, numbered.
- **Reattach instead of recreating.** Persist `session.id` and reattach on the next run rather than creating a new session each time.